In [1]:
# FLAML이라는 AutoML 라이브러리를 설치하는 코드. AutoML은 사람이 직접 여러 모델을 하나하나 돌리지 않아도, 라이브러리가 자동으로 여러 모델과 하이퍼파라미터를 찾아주는 방식. 
!pip install "flaml[automl]" --quiet

  You can safely remove it manually.


In [2]:
# 데이터 파일을 불러오고, y 에는 label을 넣고, x에는 descriptor 넣음. 
import pandas as pd
import warnings
import joblib

from sklearn.feature_selection import SelectKBest, f_classif
from flaml import AutoML

warnings.filterwarnings('ignore')


df = pd.read_csv('skin_irritation_2Ddesc.csv')

y = df['label']
X = df.drop(columns=['Chemical_Name', 'standardized_smi', 'label'])

X = X.dropna(axis=1)
X = X.loc[:, X.std() >= 0.01]

selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)

cols = X.columns[selector.get_support()]
X_sel = X[cols]

print('X_sel shape:', X_sel.shape)

X_sel shape: (39, 10)


C:\Users\DS\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 상위 10개 k만 구하기
selector = SelectKBest(f_classif, k=10)
selector.fit(X, y)

cols = X.columns[selector.get_support()]
X_sel = X[cols]

In [5]:
# FLAML에게 이 데이터로 분류 모델을 만들어줘. 30초 안에 여러 모델을 비교해서 제일 좋은 걸 찾아달라고 하는거임
automl = AutoML()

automl.fit(
    X_sel,
    y,
    task='classification',
    time_budget=30,
    eval_method='cv',
    verbose=0
)

In [6]:
# FLAML이 찾은 가장 좋은 모델을 확인하는 부분 ! 
print('최고 모델:', automl.best_estimator)
print('최적 hyperparameter:', automl.best_config)
print('CV5 정확도:', round(1 - automl.best_loss, 3))

최고 모델: xgboost
최적 hyperparameter: {'n_estimators': 4, 'max_leaves': 7, 'min_child_weight': np.float64(0.04543595007880095), 'learning_rate': np.float64(0.7243425851330955), 'subsample': 1.0, 'colsample_bylevel': 1.0, 'colsample_bytree': np.float64(0.8462712092193052), 'reg_alpha': np.float64(0.00335931099870764), 'reg_lambda': np.float64(0.11238672067536316)}
CV5 정확도: 1.0


In [7]:
# 예측 결과 확인하기
y_pred = automl.predict(X_sel)
y_prob = automl.predict_proba(X_sel)

In [8]:
# 모델별 성능 비교. 최종 1등 모델 말고 다른것들도 알려줘
for est in automl.estimator_list:
    loss = automl.best_loss_per_estimator.get(est, None)


In [18]:
# 시간을 늘려서 다시 학습해보기 . 30초였는데 60초로 늘려서 다시 해보기
automl_long = AutoML()

automl_long.fit(
    X_sel,
    y,
    task='classification',
    time_budget=60,
    eval_method='cv',
    verbose=0
)

print('30초 최고 모델:', automl.best_estimator)
print('30초 CV5 정확도:', round(1 - automl.best_loss, 3))

print('60초 최고 모델:', automl_long.best_estimator)
print('60초 CV5 정확도:', round(1 - automl_long.best_loss, 3))

30초 최고 모델: xgboost
30초 CV5 정확도: 1.0
60초 최고 모델: xgboost
60초 CV5 정확도: 1.0


In [20]:
# k=20 descriptor 선택
selector20 = SelectKBest(f_classif, k=20)

selector20.fit(X, y)

cols20 = X.columns[selector20.get_support()]
X_sel20 = X[cols20]

print('X_sel20 shape:', X_sel20.shape)


# 새로운 AutoML 객체 생성
automl_k20 = AutoML()

# 학습
automl_k20.fit(
    X_sel20,
    y,
    task='classification',
    time_budget=30,
    eval_method='cv',
    verbose=0
)


# 비교 출력
print('========== 결과 비교 ==========')

print('k=10 최고 모델:')
print(automl.best_estimator)

print('k=10 CV5 정확도:')
print(round(1 - automl.best_loss, 3))

print()

print('k=20 최고 모델:')
print(automl_k20.best_estimator)

print('k=20 CV5 정확도:')
print(round(1 - automl_k20.best_loss, 3))

X_sel20 shape: (39, 20)
========== 결과 비교 ==========
k=10 최고 모델:
xgboost
k=10 CV5 정확도:
1.0

k=20 최고 모델:
xgboost
k=20 CV5 정확도:
1.0


In [21]:
saved = {
    'features': list(cols),
    'model': automl
}

joblib.dump(saved, 'model_automl.joblib')

['model_automl.joblib']

In [22]:
loaded = joblib.load('model_automl.joblib')

features = loaded['features']
model = loaded['model']

In [30]:
from rdkit import Chem
from rdkit.Chem import Descriptors
import pandas as pd

new_mols = {
    'Acetaminophen': 'CC(=O)Nc1ccc(O)cc1',
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C'
}

rows = []

for name, smi in new_mols.items():
    mol = Chem.MolFromSmiles(smi)
    desc = Descriptors.CalcMolDescriptors(mol)
    rows.append(desc)

new_X = pd.DataFrame(rows, index=new_mols.keys())

new_X_sel = new_X[features]

new_pred = model.predict(new_X_sel)
new_prob = model.predict_proba(new_X_sel)

print('========== 새로운 화합물 예측 결과 ==========')

for i, name in enumerate(new_mols.keys()):
    print()
    print('화합물:', name)
    print('예측 label:', new_pred[i])
    print('안전(0) 확률:', round(new_prob[i][0], 3))
    print('독성(1) 확률:', round(new_prob[i][1], 3))

========== 새로운 화합물 예측 결과 ==========

화합물: Acetaminophen
예측 label: 0
안전(0) 확률: 0.978
독성(1) 확률: 0.022

화합물: Aspirin
예측 label: 0
안전(0) 확률: 0.978
독성(1) 확률: 0.022

화합물: Caffeine
예측 label: 0
안전(0) 확률: 0.884
독성(1) 확률: 0.116
